# P5.1b smoke — 100M points in density-only mode

Synthetic 100M-point Gaussian mixture streamed via the chunked Arrow IPC
protocol. `render_mode='density-only'` drops the positions buffer after
the bin grid is built, so peak GPU memory stays small (~10 MB) even at
this scale. Lasso + hover are disabled in this mode.

Memory note: Python keeps the staged numpy arrays in RAM until `_push`
completes (≈1.2 GB at N=100M). The JS side never holds more than one
chunk's positions at a time.

In [1]:
import numpy as np
from stipple import Stipple
import stipple
print(f'stipple {stipple.__version__}')

stipple 0.1.0


In [2]:
rng = np.random.default_rng(101)
n = 100_000_000
centers = np.array(
    [[-3.0, 0.0], [0.0, 2.5], [3.0, 0.0], [-1.5, -2.0], [1.5, -2.0]],
    dtype=np.float32,
)
code = rng.integers(0, 5, size=n)
xs = centers[code, 0] + rng.standard_normal(n).astype(np.float32) * 0.7
ys = centers[code, 1] + rng.standard_normal(n).astype(np.float32) * 0.7
print(f'{n:,} points · 5 clusters · {xs.nbytes/1e9:.2f} GB f32 xs')

100,000,000 points · 5 clusters · 0.40 GB f32 xs


In [3]:
w = Stipple(x=xs, y=ys, render_mode='density-only')
w

In [4]:
import time
for _ in range(600):
    if w.last_fps:
        break
    time.sleep(0.2)
print(f'status      : {w.status}')
print(f'render_mode : {w.render_mode}')
print(f'rows        : {w.rows_received:,}')
print(f'FPS         : {w.last_fps:.1f}')
print(f'frame_ms    : {w.avg_frame_ms:.2f}')
print(f'bytes_recv  : {w.bytes_received/1e6:.1f} MB')
print(f'progressive  : {w.progressive_renders}')


status      : data-rendered
render_mode : density-only
rows        : 100,000,000
FPS         : 60.0
frame_ms    : 16.65
bytes_recv  : 1200.0 MB
progressive  : 7
